# 📰 News Summarizer — Multi-Provider API Lab

**Scenario:** You're a developer at a news analytics company. This notebook builds a production-ready pipeline that:
- Fetches news articles from NewsAPI
- Uses **OpenAI** to generate summaries (fast & cheap)
- Uses **Anthropic Claude** to analyze sentiment (better at nuance)
- Has **fallback logic** if a provider fails
- **Tracks costs** to stay within budget
- **Respects rate limits**

---
### Prerequisites
You will need API keys for:
- [OpenAI](https://platform.openai.com)
- [Anthropic](https://console.anthropic.com)
- [NewsAPI](https://newsapi.org) (free tier available)

Set them in a `.env` file (see **Part 1**) or enter them when prompted.

---
## Part 1 — Setup & Configuration

In [1]:
# Install all required dependencies
!pip install openai anthropic requests python-dotenv aiohttp tiktoken pytest -q

In [2]:
"""
config.py — Configuration management for news summarizer.
Loads environment variables and exposes them as a validated Config class.
"""
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv(override=True)


class Config:
    """Application configuration."""

    # API Keys
    OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")
    ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
    NEWS_API_KEY      = os.getenv("NEWS_API_KEY")

    # Environment
    ENVIRONMENT = os.getenv("ENVIRONMENT", "development")

    # API Configuration
    MAX_RETRIES      = int(os.getenv("MAX_RETRIES", "3"))
    REQUEST_TIMEOUT  = int(os.getenv("REQUEST_TIMEOUT", "30"))

    # Models
    OPENAI_MODEL    = "gpt-4o-mini"
    ANTHROPIC_MODEL = "claude-opus-4-7"

    # Cost Control
    DAILY_BUDGET = float(os.getenv("DAILY_BUDGET", "5.00"))

    # Rate Limits (requests per minute)
    OPENAI_RPM   = 500
    ANTHROPIC_RPM = 50
    NEWS_API_RPM  = 100

    @classmethod
    def validate(cls):
        """Validate that required configuration is present."""
        required = [
            ("OPENAI_API_KEY",    cls.OPENAI_API_KEY),
            ("ANTHROPIC_API_KEY", cls.ANTHROPIC_API_KEY),
            ("NEWS_API_KEY",      cls.NEWS_API_KEY),
        ]
        missing = [name for name, val in required if not val]
        if missing:
            raise ValueError(f"Missing required configuration: {', '.join(missing)}")
        print(f"✓ Configuration validated for '{cls.ENVIRONMENT}' environment")


# ✅ Checkpoint 1 — run this cell; it should print the validation line
Config.validate()

✓ Configuration validated for 'development' environment


---
## Part 2 — News API Integration

In [3]:
"""
news_api.py — Fetch articles from NewsAPI with built-in rate limiting.
"""
import time
import requests


class NewsAPI:
    """Fetch news articles from NewsAPI."""

    def __init__(self):
        self.api_key      = Config.NEWS_API_KEY
        self.base_url     = "https://newsapi.org/v2"
        self.last_call    = 0.0
        self.min_interval = 60.0 / Config.NEWS_API_RPM  # seconds between calls

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _wait_if_needed(self):
        """Enforce rate limiting by sleeping when necessary."""
        elapsed = time.time() - self.last_call
        if elapsed < self.min_interval:
            wait = self.min_interval - elapsed
            print(f"  ⏳ Rate-limiting NewsAPI — waiting {wait:.2f}s…")
            time.sleep(wait)
        self.last_call = time.time()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def fetch_top_headlines(
        self,
        category: str = "technology",
        country:  str = "us",
        max_articles: int = 5,
    ) -> list[dict]:
        """
        Fetch top headlines.

        Args:
            category:     News category (business, technology, health, general …)
            country:      Two-letter country code (us, gb …)
            max_articles: Maximum number of articles to return.

        Returns:
            List of article dicts with keys:
            title, description, content, url, source, published_at
        """
        self._wait_if_needed()

        params = {
            "apiKey":   self.api_key,
            "category": category,
            "country":  country,
            "pageSize": max_articles,
        }

        try:
            resp = requests.get(
                f"{self.base_url}/top-headlines",
                params=params,
                timeout=Config.REQUEST_TIMEOUT,
            )
            resp.raise_for_status()
            data = resp.json()

            if data.get("status") != "ok":
                raise RuntimeError(f"NewsAPI error: {data.get('message')}")

            articles = [
                {
                    "title":        a.get("title", ""),
                    "description":  a.get("description", ""),
                    "content":      a.get("content", ""),
                    "url":          a.get("url", ""),
                    "source":       a.get("source", {}).get("name", "Unknown"),
                    "published_at": a.get("publishedAt", ""),
                }
                for a in data.get("articles", [])
            ]

            print(f"✓ Fetched {len(articles)} article(s) from NewsAPI")
            return articles

        except requests.RequestException as exc:
            print(f"✗ Error fetching news: {exc}")
            return []


# ✅ Checkpoint 2 — fetch 3 technology articles
news_api = NewsAPI()
sample_articles = news_api.fetch_top_headlines(category="technology", max_articles=3)

for i, art in enumerate(sample_articles, 1):
    print(f"\n{i}. {art['title']}")
    print(f"   Source: {art['source']}")
    print(f"   URL:    {art['url']}")

✓ Fetched 3 article(s) from NewsAPI

1. Fears mount that PlayStation has "re-armed the CBOMB" issue which stops you playing purchased games, amid 30-day DRM and AI chat bot confusion - Eurogamer.net
   Source: Google News
   URL:    https://news.google.com/rss/articles/CBMieEFVX3lxTE9tRS1tbmE1Vy1MNjJzTDRKUDByeGtoeE9BeWdsQkZuQlJ5aXBBTzNXaDlEOGMzS2QzRlphWm1PamRLLWJ1SVZ2SDZPZkI4dDA0dHZyeTByQzk1UWRHTjZlS2FZWWRKeDNOVXBMbW1oZ213QWU1YlVMSg?oc=5

2. Microsoft’s Windows secrete K2 plan aims to cut Windows 11 bloat and improve gaming performance - VideoCardz.com
   Source: Google News
   URL:    https://news.google.com/rss/articles/CBMivwFBVV95cUxPMEFDVmtrMUd4WkJQREk1WWtUUUlaTzN4WlZFUk9qeDNpajJ0MnBYeEtxQjk1eHh6RTJNUENrbWNyQVBNZldyOHNpRHM3VGFkT0RHeHczcDdEdUs5RGdGZEczSmU2RVdYQmJHS1NaUWtyd2NJX2NnRkw2TmxlQmdGdFdXWWpMQlBxSUR4TzNZbG83aFNxbng4VDVqS1dqSDFzd29sYzJ4Y0c3UTczTE8yQkl3OXZRVFhNXzRzQWx1SQ?oc=5

3. "We've Heard About Stick Drift and We've Experienced It Ourselves": The Big Steam Controller Inter

---
## Part 3 — LLM Provider Integration

In [4]:
"""
llm_providers.py — OpenAI + Anthropic clients with cost tracking,
rate limiting, and automatic fallback.
"""
import time
import tiktoken
from openai    import OpenAI
from anthropic import Anthropic


# ---------------------------------------------------------------------------
# Pricing table  (USD per 1 million tokens)
# ---------------------------------------------------------------------------
PRICING = {
    "gpt-4o-mini":                       {"input": 0.15,  "output": 0.60},
    "gpt-4o":                            {"input": 2.50,  "output": 10.00},
    "claude-opus-4-7":                   {"input": 3.00,  "output": 15.00},
}


# ---------------------------------------------------------------------------
# Helper: token counting
# ---------------------------------------------------------------------------

def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    """Return the number of tokens in *text* for *model*.
    Falls back to character-based estimate when tiktoken has no encoding."""
    try:
        enc = tiktoken.encoding_for_model(model)
        return len(enc.encode(text))
    except Exception:
        return len(text) // 4  # rough fallback: ~4 chars per token


# ---------------------------------------------------------------------------
# Cost tracker
# ---------------------------------------------------------------------------

class CostTracker:
    """Accumulate per-request token usage and compute USD costs."""

    def __init__(self):
        self.total_cost = 0.0
        self.requests:  list[dict] = []

    def track_request(
        self,
        provider: str,
        model:    str,
        input_tokens:  int,
        output_tokens: int,
    ) -> float:
        """Record one API call and return its cost in USD."""
        pricing     = PRICING.get(model, {"input": 3.0, "output": 15.0})
        input_cost  = (input_tokens  / 1_000_000) * pricing["input"]
        output_cost = (output_tokens / 1_000_000) * pricing["output"]
        cost = input_cost + output_cost

        self.total_cost += cost
        self.requests.append({
            "provider":      provider,
            "model":         model,
            "input_tokens":  input_tokens,
            "output_tokens": output_tokens,
            "cost":          cost,
        })
        return cost

    def get_summary(self) -> dict:
        """Return aggregate statistics."""
        total_in  = sum(r["input_tokens"]  for r in self.requests)
        total_out = sum(r["output_tokens"] for r in self.requests)
        return {
            "total_requests":      len(self.requests),
            "total_cost":          self.total_cost,
            "total_input_tokens":  total_in,
            "total_output_tokens": total_out,
            "average_cost":        self.total_cost / max(len(self.requests), 1),
        }

    def check_budget(self, daily_budget: float) -> None:
        """Raise if the daily budget is exceeded; warn at 90 %."""
        if self.total_cost >= daily_budget:
            raise RuntimeError(
                f"Daily budget of ${daily_budget:.2f} exceeded! "
                f"Current: ${self.total_cost:.2f}"
            )
        pct = (self.total_cost / daily_budget) * 100
        if pct >= 90:
            print(f"⚠️  Warning: {pct:.1f}% of daily budget used")


# ---------------------------------------------------------------------------
# Multi-provider client
# ---------------------------------------------------------------------------

class LLMProviders:
    """Manage OpenAI and Anthropic with rate limiting, cost tracking, and fallback."""

    def __init__(self):
        self.openai_client    = OpenAI(api_key=Config.OPENAI_API_KEY)
        self.anthropic_client = Anthropic(api_key=Config.ANTHROPIC_API_KEY)
        self.cost_tracker     = CostTracker()

        # Rate-limit state
        self._openai_last    = 0.0
        self._anthropic_last = 0.0
        self._openai_interval    = 60.0 / Config.OPENAI_RPM
        self._anthropic_interval = 60.0 / Config.ANTHROPIC_RPM

    # ------------------------------------------------------------------
    # Rate-limit helpers
    # ------------------------------------------------------------------

    def _wait_openai(self):
        elapsed = time.time() - self._openai_last
        if elapsed < self._openai_interval:
            time.sleep(self._openai_interval - elapsed)
        self._openai_last = time.time()

    def _wait_anthropic(self):
        elapsed = time.time() - self._anthropic_last
        if elapsed < self._anthropic_interval:
            time.sleep(self._anthropic_interval - elapsed)
        self._anthropic_last = time.time()

    # ------------------------------------------------------------------
    # Provider calls
    # ------------------------------------------------------------------

    def ask_openai(self, prompt: str, model: str | None = None) -> str:
        """Send *prompt* to OpenAI and return the response text."""
        model = model or Config.OPENAI_MODEL
        self._wait_openai()

        in_tokens = count_tokens(prompt, model)

        response = self.openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        text      = response.choices[0].message.content
        out_tokens = count_tokens(text, model)

        cost = self.cost_tracker.track_request("openai", model, in_tokens, out_tokens)
        self.cost_tracker.check_budget(Config.DAILY_BUDGET)
        print(f"    [OpenAI] {in_tokens}→{out_tokens} tokens | ${cost:.6f}")
        return text

    def ask_anthropic(self, prompt: str, model: str | None = None) -> str:
        """Send *prompt* to Anthropic Claude and return the response text."""
        model = model or Config.ANTHROPIC_MODEL
        self._wait_anthropic()

        in_tokens = count_tokens(prompt, model)

        response = self.anthropic_client.messages.create(
            model=model,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
        )
        text      = response.content[0].text
        out_tokens = count_tokens(text, model)

        cost = self.cost_tracker.track_request("anthropic", model, in_tokens, out_tokens)
        self.cost_tracker.check_budget(Config.DAILY_BUDGET)
        print(f"    [Anthropic] {in_tokens}→{out_tokens} tokens | ${cost:.6f}")
        return text

    def ask_with_fallback(self, prompt: str, primary: str = "openai") -> dict:
        """
        Call the primary provider; fall back to the other on failure.

        Args:
            prompt:  The question / instruction text.
            primary: "openai" or "anthropic".

        Returns:
            {"provider": str, "response": str}
        """
        order = [
            (primary,                                    self.ask_openai if primary == "openai" else self.ask_anthropic),
            ("anthropic" if primary == "openai" else "openai", self.ask_anthropic if primary == "openai" else self.ask_openai),
        ]

        last_exc = None
        for provider_name, caller in order:
            try:
                print(f"  → Trying {provider_name} …")
                return {"provider": provider_name, "response": caller(prompt)}
            except Exception as exc:
                print(f"  ✗ {provider_name} failed: {exc}")
                last_exc = exc

        raise RuntimeError("All providers failed") from last_exc


# ✅ Checkpoint 3 — test both providers
llm = LLMProviders()

print("Testing OpenAI:")
r1 = llm.ask_openai("What is Python? One sentence.")
print(f"  {r1}\n")

print("Testing Anthropic:")
r2 = llm.ask_anthropic("What is Python? One sentence.")
print(f"  {r2}\n")

print("Testing fallback:")
r3 = llm.ask_with_fallback("What is machine learning? One sentence.")
print(f"  Provider used: {r3['provider']}")
print(f"  {r3['response']}\n")

s = llm.cost_tracker.get_summary()
print(f"Cumulative cost so far: ${s['total_cost']:.5f}  ({s['total_requests']} requests)")

Testing OpenAI:
    [OpenAI] 7→36 tokens | $0.000023
  Python is a high-level, interpreted programming language known for its readability, simplicity, and versatility, making it popular for a wide range of applications, from web development to data science.

Testing Anthropic:
    [Anthropic] 7→52 tokens | $0.000801
  Python is a high-level, interpreted, general-purpose programming language known for its readable syntax and versatility across domains like web development, data science, automation, and artificial intelligence.

Testing fallback:
  → Trying openai …
    [OpenAI] 8→26 tokens | $0.000017
  Provider used: openai
  Machine learning is a branch of artificial intelligence that enables systems to learn from data and improve their performance over time without being explicitly programmed.

Cumulative cost so far: $0.00084  (3 requests)


---
## Part 4 — News Summarizer Core Logic

In [5]:
"""
summarizer.py — Orchestrates NewsAPI → OpenAI (summary) → Anthropic (sentiment)
with per-article error handling and a final cost report.
"""


class NewsSummarizer:
    """Summarize news articles using multiple LLM providers."""

    def __init__(self):
        self.news_api      = NewsAPI()
        self.llm_providers = LLMProviders()

    # ------------------------------------------------------------------
    # Single-article pipeline
    # ------------------------------------------------------------------

    def summarize_article(self, article: dict) -> dict:
        """
        Run the full pipeline for one article.

        Steps:
          1. Build the article text snippet.
          2. Ask OpenAI for a 2–3 sentence summary (with Anthropic fallback).
          3. Ask Anthropic for sentiment analysis of the summary.

        Args:
            article: Dict with keys title, description, content, url, source, published_at.

        Returns:
            Dict with title, source, url, summary, sentiment, published_at.
        """
        title = article['title'] or '(no title)'
        print(f"\nProcessing: {title[:70]}…")

        article_text = (
            f"Title: {article['title']}\n"
            f"Description: {article['description']}\n"
            f"Content: {(article['content'] or '')[:500]}"
        )

        # ---- Step 1: Summarisation (OpenAI primary, Anthropic fallback) ----
        summary_prompt = (
            f"Summarize this news article in 2–3 sentences:\n\n{article_text}"
        )
        try:
            print("  → Summarising with OpenAI…")
            summary = self.llm_providers.ask_openai(summary_prompt)
            print("  ✓ Summary generated")
        except Exception as exc:
            print(f"  ✗ OpenAI failed: {exc} — falling back to Anthropic…")
            summary = self.llm_providers.ask_anthropic(summary_prompt)
            print("  ✓ Summary generated (via fallback)")

        # ---- Step 2: Sentiment analysis (Anthropic — better at nuance) ----
        sentiment_prompt = (
            f'Analyse the sentiment of this text: "{summary}"\n\n'
            "Provide:\n"
            "- Overall sentiment (positive / negative / neutral)\n"
            "- Confidence (0–100 %)\n"
            "- Key emotional tone\n\n"
            "Be concise (2–3 sentences)."
        )
        try:
            print("  → Analysing sentiment with Anthropic…")
            sentiment = self.llm_providers.ask_anthropic(sentiment_prompt)
            print("  ✓ Sentiment analysed")
        except Exception as exc:
            print(f"  ✗ Anthropic sentiment failed: {exc}")
            sentiment = "Unable to analyse sentiment."

        return {
            "title":        article["title"],
            "source":       article["source"],
            "url":          article["url"],
            "summary":      summary,
            "sentiment":    sentiment,
            "published_at": article["published_at"],
        }

    # ------------------------------------------------------------------
    # Batch processing
    # ------------------------------------------------------------------

    def process_articles(self, articles: list[dict]) -> list[dict]:
        """Process a list of articles, skipping those that raise errors."""
        results = []
        for article in articles:
            try:
                results.append(self.summarize_article(article))
            except Exception as exc:
                print(f"  ✗ Skipping article due to error: {exc}")
        return results

    # ------------------------------------------------------------------
    # Reporting
    # ------------------------------------------------------------------

    def generate_report(self, results: list[dict]) -> None:
        """Print a formatted report of all processed articles."""
        sep = "=" * 80
        print(f"\n{sep}")
        print("NEWS SUMMARY REPORT")
        print(sep)

        for i, res in enumerate(results, 1):
            print(f"\n{i}. {res['title']}")
            print(f"   Source: {res['source']}  |  Published: {res['published_at']}")
            print(f"   URL: {res['url']}")
            print(f"\n   SUMMARY:\n   {res['summary']}")
            print(f"\n   SENTIMENT:\n   {res['sentiment']}")
            print(f"\n   {'-'*76}")

        summary = self.llm_providers.cost_tracker.get_summary()
        print(f"\n{sep}")
        print("COST SUMMARY")
        print(sep)
        print(f"  Total requests : {summary['total_requests']}")
        print(f"  Total cost     : ${summary['total_cost']:.5f}")
        print(f"  Total tokens   : {summary['total_input_tokens'] + summary['total_output_tokens']:,}")
        print(f"    Input        : {summary['total_input_tokens']:,}")
        print(f"    Output       : {summary['total_output_tokens']:,}")
        print(f"  Avg cost/req   : ${summary['average_cost']:.7f}")
        print(sep)


# ✅ Checkpoint 4 — fetch 2 articles and run the full pipeline
summarizer = NewsSummarizer()

print("Fetching articles…")
articles = summarizer.news_api.fetch_top_headlines(category="technology", max_articles=2)

if not articles:
    print("No articles fetched — check your NEWS_API_KEY.")
else:
    results = summarizer.process_articles(articles)
    summarizer.generate_report(results)

Fetching articles…
✓ Fetched 2 article(s) from NewsAPI

Processing: Fears mount that PlayStation has "re-armed the CBOMB" issue which stop…
  → Summarising with OpenAI…
    [OpenAI] 61→79 tokens | $0.000057
  ✓ Summary generated
  → Analysing sentiment with Anthropic…
    [Anthropic] 149→48 tokens | $0.001167
  ✓ Sentiment analysed

Processing: Microsoft’s Windows secrete K2 plan aims to cut Windows 11 bloat and i…
  → Summarising with OpenAI…
    [OpenAI] 48→62 tokens | $0.000044
  ✓ Summary generated
  → Analysing sentiment with Anthropic…
    [Anthropic] 140→56 tokens | $0.001260
  ✓ Sentiment analysed

NEWS SUMMARY REPORT

1. Fears mount that PlayStation has "re-armed the CBOMB" issue which stops you playing purchased games, amid 30-day DRM and AI chat bot confusion - Eurogamer.net
   Source: Google News  |  Published: 2026-04-29T12:49:15Z
   URL: https://news.google.com/rss/articles/CBMieEFVX3lxTE9tRS1tbmE1Vy1MNjJzTDRKUDByeGtoeE9BeWdsQkZuQlJ5aXBBTzNXaDlEOGMzS2QzRlphWm1PamRLLWJ1SVZ

---
## Part 5 — Async Processing (Optional Advanced Feature)

In [8]:
"""
AsyncNewsSummarizer — processes multiple articles concurrently using
asyncio.to_thread so the (sync) LLM calls do not block each other.
"""
import asyncio


class AsyncNewsSummarizer(NewsSummarizer):
    """Async subclass for concurrent article processing."""

    async def summarize_article_async(self, article: dict) -> dict:
        """Wrap the synchronous pipeline so it can run in a thread pool."""
        return await asyncio.to_thread(self.summarize_article, article)

    async def process_articles_async(
        self,
        articles: list[dict],
        max_concurrent: int = 3,
    ) -> list[dict]:
        """
        Process articles with a concurrency cap.

        Args:
            articles:       Articles to process.
            max_concurrent: Max parallel workers (default 3).

        Returns:
            List of successfully processed results.
        """
        sem = asyncio.Semaphore(max_concurrent)

        async def _guarded(art):
            async with sem:
                return await self.summarize_article_async(art)

        tasks   = [_guarded(a) for a in articles]
        raw     = await asyncio.gather(*tasks, return_exceptions=True)
        results = [r for r in raw if not isinstance(r, Exception)]
        errors  = [r for r in raw if isinstance(r, Exception)]

        if errors:
            print(f"⚠️  {len(errors)} article(s) failed during async processing.")

        return results


# ✅ Optional: run async pipeline on 5 articles
async def demo_async():
    async_summarizer = AsyncNewsSummarizer()
    print("Fetching 5 technology articles…")
    arts = async_summarizer.news_api.fetch_top_headlines(
        category="technology", max_articles=5
    )
    if arts:
        print(f"Processing {len(arts)} articles concurrently (max 3 at a time)…")
        res = await async_summarizer.process_articles_async(arts, max_concurrent=3)
        async_summarizer.generate_report(res)


# Uncomment the line below to run the async demo:
await demo_async()

Fetching 5 technology articles…
✓ Fetched 5 article(s) from NewsAPI
Processing 5 articles concurrently (max 3 at a time)…

Processing: Fears mount that PlayStation has "re-armed the CBOMB" issue which stop…
  → Summarising with OpenAI…

Processing: Microsoft’s Windows secrete K2 plan aims to cut Windows 11 bloat and i…
  → Summarising with OpenAI…

Processing: "We've Heard About Stick Drift and We've Experienced It Ourselves": Th…
  → Summarising with OpenAI…
    [OpenAI] 61→76 tokens | $0.000055
  ✓ Summary generated
  → Analysing sentiment with Anthropic…
    [OpenAI] 48→76 tokens | $0.000053
  ✓ Summary generated
  → Analysing sentiment with Anthropic…
    [OpenAI] 46→77 tokens | $0.000053
  ✓ Summary generated
  → Analysing sentiment with Anthropic…
    [Anthropic] 152→53 tokens | $0.001251
  ✓ Sentiment analysed

Processing: John Ternus faces critical decisions on iPhone pricing and US manufact…
  → Summarising with OpenAI…
    [Anthropic] 154→70 tokens | $0.001512
  ✓ Sentiment a

---
## Part 6 — Unit Tests

In [10]:
"""
test_summarizer.py — Unit tests using pytest + unittest.mock.
Run directly in this cell; results are printed inline.
"""
from unittest.mock import Mock, patch, MagicMock


# ---------------------------------------------------------------------------
# CostTracker tests
# ---------------------------------------------------------------------------

class TestCostTracker:

    def test_track_single_request(self):
        tracker = CostTracker()
        cost = tracker.track_request("openai", "gpt-4o-mini", 100, 500)
        assert cost > 0, "Cost must be positive"
        assert abs(tracker.total_cost - cost) < 1e-9
        assert len(tracker.requests) == 1
        print("  ✓ test_track_single_request")

    def test_get_summary(self):
        tracker = CostTracker()
        tracker.track_request("openai",    "gpt-4o-mini",               100, 200)
        tracker.track_request("anthropic", "claude-opus-4-7",           150, 300)
        s = tracker.get_summary()
        assert s["total_requests"]      == 2
        assert s["total_cost"]          >  0
        assert s["total_input_tokens"]  == 250
        assert s["total_output_tokens"] == 500
        print("  ✓ test_get_summary")

    def test_budget_ok(self):
        tracker = CostTracker()
        tracker.track_request("openai", "gpt-4o-mini", 100, 100)
        tracker.check_budget(10.00)   # should NOT raise
        print("  ✓ test_budget_ok")

    def test_budget_exceeded(self):
        tracker = CostTracker()
        tracker.total_cost = 15.00
        raised = False
        try:
            tracker.check_budget(10.00)
        except RuntimeError as e:
            assert "exceeded" in str(e).lower()
            raised = True
        assert raised, "Should have raised RuntimeError"
        print("  ✓ test_budget_exceeded")


# ---------------------------------------------------------------------------
# Token counting test
# ---------------------------------------------------------------------------

class TestTokenCounting:

    def test_count_tokens_basic(self):
        n = count_tokens("Hello, how are you?")
        assert 0 < n < 20
        print("  ✓ test_count_tokens_basic")


# ---------------------------------------------------------------------------
# NewsAPI tests
# ---------------------------------------------------------------------------

class TestNewsAPI:

    def test_fetch_top_headlines_success(self):
        mock_resp = Mock()
        mock_resp.status_code = 200
        mock_resp.json.return_value = {
            "status": "ok",
            "articles": [{
                "title":       "Test Article",
                "description": "Test description",
                "content":     "Test content",
                "url":         "https://example.com",
                "source":      {"name": "Test Source"},
                "publishedAt": "2026-04-30",
            }],
        }
        mock_resp.raise_for_status = Mock()

        with patch("requests.get", return_value=mock_resp):
            api   = NewsAPI()
            arts  = api.fetch_top_headlines(max_articles=1)

        assert len(arts)         == 1
        assert arts[0]["title"]  == "Test Article"
        assert arts[0]["source"] == "Test Source"
        print("  ✓ test_fetch_top_headlines_success")

    def test_fetch_top_headlines_network_error(self):
        import requests as req_lib
        with patch("requests.get", side_effect=req_lib.RequestException("timeout")):
            api  = NewsAPI()
            arts = api.fetch_top_headlines(max_articles=1)
        assert arts == []
        print("  ✓ test_fetch_top_headlines_network_error")


# ---------------------------------------------------------------------------
# LLMProviders tests
# ---------------------------------------------------------------------------

class TestLLMProviders:

    def _make_openai_mock(self, text="Test response"):
        mock_client   = MagicMock()
        mock_response = MagicMock()
        mock_response.choices = [MagicMock(message=MagicMock(content=text))]
        mock_client.chat.completions.create.return_value = mock_response
        return mock_client

    def _make_anthropic_mock(self, text="Test sentiment"):
        mock_client   = MagicMock()
        mock_response = MagicMock()
        mock_response.content = [MagicMock(text=text)]
        mock_client.messages.create.return_value = mock_response
        return mock_client

    def test_ask_openai(self):
        providers = LLMProviders()
        providers.openai_client = self._make_openai_mock("Python is great!")
        resp = providers.ask_openai("What is Python?")
        assert resp == "Python is great!"
        print("  ✓ test_ask_openai")

    def test_ask_anthropic(self):
        providers = LLMProviders()
        providers.anthropic_client = self._make_anthropic_mock("Very positive.")
        resp = providers.ask_anthropic("Analyse sentiment.")
        assert resp == "Very positive."
        print("  ✓ test_ask_anthropic")

    def test_fallback_triggers_on_openai_failure(self):
        providers = LLMProviders()
        # OpenAI raises, Anthropic succeeds
        providers.openai_client    = MagicMock()
        providers.openai_client.chat.completions.create.side_effect = RuntimeError("quota")
        providers.anthropic_client = self._make_anthropic_mock("Fallback response")

        result = providers.ask_with_fallback("Some question", primary="openai")
        assert result["provider"]  == "anthropic"
        assert result["response"]  == "Fallback response"
        print("  ✓ test_fallback_triggers_on_openai_failure")


# ---------------------------------------------------------------------------
# NewsSummarizer tests
# ---------------------------------------------------------------------------

class TestNewsSummarizer:

    def test_initialization(self):
        s = NewsSummarizer()
        assert s.news_api      is not None
        assert s.llm_providers is not None
        print("  ✓ test_initialization")

    def test_summarize_article(self):
        s = NewsSummarizer()

        # Patch ask_openai and ask_anthropic on the instance
        s.llm_providers.openai_client    = MagicMock()
        s.llm_providers.anthropic_client = MagicMock()

        mock_oa_resp = MagicMock()
        mock_oa_resp.choices = [MagicMock(message=MagicMock(content="Great summary"))]
        s.llm_providers.openai_client.chat.completions.create.return_value = mock_oa_resp

        mock_an_resp = MagicMock()
        mock_an_resp.content = [MagicMock(text="Positive. 90% confidence.")]
        s.llm_providers.anthropic_client.messages.create.return_value = mock_an_resp

        article = {
            "title":        "AI Breakthrough",
            "description":  "Scientists announce major AI achievement.",
            "content":      "Details here…",
            "url":          "https://example.com/ai",
            "source":       "Tech Times",
            "published_at": "2026-04-30",
        }

        result = s.summarize_article(article)
        assert result["title"]     == "AI Breakthrough"
        assert result["summary"]   == "Great summary"
        assert "Positive" in result["sentiment"]
        print("  ✓ test_summarize_article")


# ---------------------------------------------------------------------------
# Run all tests
# ---------------------------------------------------------------------------

def run_all_tests():
    suites = [
        TestCostTracker,
        TestTokenCounting,
        TestNewsAPI,
        TestLLMProviders,
        TestNewsSummarizer,
    ]

    total = passed = failed = 0
    for Suite in suites:
        print(f"\n{'─'*60}")
        print(f"  {Suite.__name__}")
        print(f"{'─'*60}")
        inst = Suite()
        for name in [m for m in dir(inst) if m.startswith("test_")]:
            total += 1
            try:
                getattr(inst, name)()
                passed += 1
            except Exception as exc:
                failed += 1
                print(f"  ✗ {name}: {exc}")

    print(f"\n{'='*60}")
    print(f"Results: {passed}/{total} passed, {failed} failed")
    print("='*60")
    return failed == 0


# ✅ Checkpoint 5 — run unit tests
run_all_tests()


────────────────────────────────────────────────────────────
  TestCostTracker
────────────────────────────────────────────────────────────
  ✓ test_budget_exceeded
  ✓ test_budget_ok
  ✓ test_get_summary
  ✓ test_track_single_request

────────────────────────────────────────────────────────────
  TestTokenCounting
────────────────────────────────────────────────────────────
  ✓ test_count_tokens_basic

────────────────────────────────────────────────────────────
  TestNewsAPI
────────────────────────────────────────────────────────────
✗ Error fetching news: timeout
  ✓ test_fetch_top_headlines_network_error
✓ Fetched 1 article(s) from NewsAPI
  ✓ test_fetch_top_headlines_success

────────────────────────────────────────────────────────────
  TestLLMProviders
────────────────────────────────────────────────────────────
    [Anthropic] 4→3 tokens | $0.000057
  ✓ test_ask_anthropic
    [OpenAI] 4→4 tokens | $0.000003
  ✓ test_ask_openai
  → Trying openai …
  ✗ openai failed: quota
  → 

True

---
## Part 7 — Interactive Main Application

In [11]:
"""
main.py — Interactive entry point.
Configure the variables below and run the cell.
"""

# ── Configuration ────────────────────────────────────────────────────────────
CATEGORY     = "technology"   # technology | business | health | general
NUM_ARTICLES = 3              # 1 – 10
USE_ASYNC    = False          # True → concurrent processing
# ─────────────────────────────────────────────────────────────────────────────

import asyncio

print("=" * 80)
print("NEWS SUMMARIZER — Multi-Provider Edition")
print("=" * 80)
print(f"Category : {CATEGORY}")
print(f"Articles : {NUM_ARTICLES}")
print(f"Async    : {USE_ASYNC}")
print()

NUM_ARTICLES = max(1, min(10, NUM_ARTICLES))

try:
    if USE_ASYNC:
        async_s = AsyncNewsSummarizer()
        arts = async_s.news_api.fetch_top_headlines(
            category=CATEGORY, max_articles=NUM_ARTICLES
        )
        if arts:
            res = await async_s.process_articles_async(arts, max_concurrent=3)
            async_s.generate_report(res)
    else:
        sync_s = NewsSummarizer()
        arts = sync_s.news_api.fetch_top_headlines(
            category=CATEGORY, max_articles=NUM_ARTICLES
        )
        if arts:
            res = sync_s.process_articles(arts)
            sync_s.generate_report(res)

    print("\n✓ Processing complete!")

except Exception as e:
    print(f"\n✗ Error: {e}")

NEWS SUMMARIZER — Multi-Provider Edition
Category : technology
Articles : 3
Async    : False

✓ Fetched 3 article(s) from NewsAPI

Processing: Fears mount that PlayStation has "re-armed the CBOMB" issue which stop…
  → Summarising with OpenAI…
    [OpenAI] 61→76 tokens | $0.000055
  ✓ Summary generated
  → Analysing sentiment with Anthropic…
    [Anthropic] 148→58 tokens | $0.001314
  ✓ Sentiment analysed

Processing: Microsoft’s Windows secrete K2 plan aims to cut Windows 11 bloat and i…
  → Summarising with OpenAI…
    [OpenAI] 48→64 tokens | $0.000046
  ✓ Summary generated
  → Analysing sentiment with Anthropic…
    [Anthropic] 129→77 tokens | $0.001542
  ✓ Sentiment analysed

Processing: "We've Heard About Stick Drift and We've Experienced It Ourselves": Th…
  → Summarising with OpenAI…
    [OpenAI] 46→71 tokens | $0.000049
  ✓ Summary generated
  → Analysing sentiment with Anthropic…
    [Anthropic] 147→71 tokens | $0.001506
  ✓ Sentiment analysed

NEWS SUMMARY REPORT

1. Fears mo

---
## Lab Summary

| Objective | Where implemented |
|---|---|
| Fetch from external API | `NewsAPI.fetch_top_headlines` (Part 2) |
| OpenAI summarisation | `LLMProviders.ask_openai` (Part 3) |
| Anthropic sentiment | `LLMProviders.ask_anthropic` (Part 3) |
| Fallback logic | `LLMProviders.ask_with_fallback` (Part 3) |
| Cost tracking | `CostTracker` class (Part 3) |
| Rate limiting | `_wait_openai / _wait_anthropic / _wait_if_needed` (Parts 2–3) |
| Unit tests | `TestCostTracker` … `TestNewsSummarizer` (Part 6) |
| Async concurrency | `AsyncNewsSummarizer` (Part 5) |

**Cost estimate for 500 articles/day** (assuming ~300 input + 150 output tokens/article × 2 calls):
- OpenAI gpt-4o-mini: ~$0.023/day
- Anthropic claude-opus-4-7: ~$0.68/day
- **Total ≈ $0.70/day** — well within the $5.00 budget.